# BUSM131 — Telco Customer Churn: Exploratory Data Analysis

Comprehensive EDA on the IBM Telco Customer Churn dataset (7,043 customers, 21 features).

**Contents:** Data cleaning, univariate & bivariate statistics, visualisations (distributions, boxplots, churn by category, correlation heatmap, tenure analysis).


In [ ]:
%matplotlib inline


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 1. LOAD & INSPECT


In [ ]:
print("=" * 65)


In [ ]:
print("=" * 65)

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nDtypes:\n{df.dtypes}")


## 2. DATA CLEANING


In [ ]:
print("\n" + "-" * 65)


In [ ]:
print("-" * 65)

# TotalCharges has blank strings → convert
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing = df['TotalCharges'].isnull().sum()
print(f"TotalCharges missing: {missing} (new customers with tenure=0)")
print(df[df['TotalCharges'].isnull()][['customerID','tenure','MonthlyCharges']].head())
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f"Duplicates: {df.duplicated().sum()}")
print(f"Missing after cleaning:\n{df.isnull().sum()[df.isnull().sum()>0]}")
if df.isnull().sum().sum() == 0:
    print("  None — dataset is clean.")


## 3. UNIVARIATE DESCRIPTIVE STATISTICS


In [ ]:
print("\n" + "-" * 65)


## UNIVARIATE STATISTICS — NUMERICAL


In [ ]:
print("-" * 65)
print(df[['tenure','MonthlyCharges','TotalCharges']].describe().round(2))

print("\n" + "-" * 65)


## TARGET VARIABLE — Churn


In [ ]:
print("-" * 65)
churn_ct = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True)*100
print(pd.DataFrame({'Count': churn_ct, '%': churn_pct.round(2)}))


## 4. VISUALISATIONS


In [ ]:
colors = ['#2ecc71','#e74c3c']


**Fig 1: Churn distribution**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
churn_ct.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Churn Distribution (Count)', fontweight='bold')
axes[0].set_xticklabels(['No','Yes'], rotation=0)
axes[0].set_ylabel('Count')
axes[1].pie(churn_ct, labels=['No','Yes'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0,0.05))
axes[1].set_title('Churn Distribution (%)', fontweight='bold')
plt.tight_layout()
plt.show()


**Fig 2: Numerical distributions by Churn**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['tenure','MonthlyCharges','TotalCharges']):
    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=axes[i],
                 palette={'No':'#2ecc71','Yes':'#e74c3c'}, alpha=0.6)
    axes[i].set_title(f'{col} by Churn', fontweight='bold')
plt.tight_layout()
plt.show()


**Fig 3: Boxplots**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['tenure','MonthlyCharges','TotalCharges']):
    sns.boxplot(data=df, x='Churn', y=col, ax=axes[i],
                palette={'No':'#2ecc71','Yes':'#e74c3c'})
    axes[i].set_title(f'{col} by Churn', fontweight='bold')
plt.tight_layout()
plt.show()


**Fig 4: Churn rate by key categorical features**


In [ ]:
key_cats = ['Contract','InternetService','PaymentMethod','TechSupport',
            'OnlineSecurity','OnlineBackup','PaperlessBilling',
            'SeniorCitizen','Partner','Dependents','gender','PhoneService']
fig, axes = plt.subplots(3, 4, figsize=(22, 15))
axes = axes.flatten()
for i, col in enumerate(key_cats):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index')*100
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=colors, edgecolor='black')
    axes[i].set_title(f'Churn by {col}', fontweight='bold', fontsize=11)
    axes[i].set_ylabel('%'); axes[i].set_xlabel('')
    axes[i].legend(title='Churn', loc='upper right', fontsize=8)
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


**Fig 5: Correlation heatmap**


In [ ]:
df_enc = df.copy()
binary_map = {'Yes':1,'No':0,'Male':1,'Female':0}
for col in df_enc.select_dtypes(include='object').columns:
    if set(df_enc[col].unique()).issubset({'Yes','No'}):
        df_enc[col] = df_enc[col].map(binary_map)
    elif col == 'gender':
        df_enc[col] = df_enc[col].map(binary_map)
num_cols = df_enc.select_dtypes(include=[np.number]).columns
corr = df_enc[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


**Fig 6: Churn by tenure groups**


In [ ]:
df['tenure_group'] = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                            labels=['0-12','13-24','25-48','49-72'])
ct = pd.crosstab(df['tenure_group'], df['Churn'], normalize='index')*100
fig, ax = plt.subplots(figsize=(10, 6))
ct.plot(kind='bar', stacked=True, color=colors, edgecolor='black', ax=ax)
ax.set_title('Churn Rate by Tenure Group (months)', fontweight='bold')
ax.set_ylabel('%'); plt.xticks(rotation=0)
ax.legend(title='Churn')
plt.tight_layout()
plt.show()
df.drop('tenure_group', axis=1, inplace=True)


## 5. BIVARIATE STATISTICS


In [ ]:
print("\n" + "-" * 65)


In [ ]:
print("-" * 65)
print("\nMean by Churn:")
print(df.groupby('Churn')[['tenure','MonthlyCharges','TotalCharges']].mean().round(2))

print("\nChurn rate by Contract:")
print((pd.crosstab(df['Contract'], df['Churn'], normalize='index')*100).round(1))

print("\nChurn rate by Internet Service:")
print((pd.crosstab(df['InternetService'], df['Churn'], normalize='index')*100).round(1))

print("\nChurn rate by Payment Method:")
print((pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index')*100).round(1))


## 6. KEY INSIGHTS


In [ ]:
print("\n" + "=" * 65)


## KEY EDA INSIGHTS


In [ ]:
print("=" * 65)
print("""
1. CLASS IMBALANCE: 26.5% churn vs 73.5% retained (moderate).
2. TENURE: Churners avg ~18 months vs ~38 months for retained.
3. MONTHLY CHARGES: Churners pay more (~£74 vs ~£61).
4. CONTRACT: Month-to-month → 42.7% churn; Two-year → 2.8%.
5. INTERNET: Fiber optic → 41.9% churn vs 19% for DSL.
6. PAYMENT: Electronic check → 45.3% churn rate (highest).
7. SERVICES: No online security / tech support → higher churn.
8. SENIOR CITIZENS: Higher churn (~42% vs ~24%).
9. GENDER: No significant difference.
""")

# Save cleaned version
